# ML-02 — Research Question and Provisional Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

**Provisional lane: Lane 2 — Refresh / Content Opportunity Scoring.**

I'm picking this lane because the starter dataset already shows a large, messy pool of pages that
look like candidates for review, but no obvious single rule separates the ones worth an editor's
time from the ones that are fine as-is. A ranked, evidence-backed queue is a natural fit: FlyRank's
own product already frames the problem this way (a `refresh_queue`), and the starter pipeline
(`scripts/01`-`05`) exists specifically to build and score one. Section 3 below backs this with
real counts from `data/raw/content_refresh_anonymized.csv`. This is provisional — I can confirm or
switch through the end of Week 4.

## 2. The question: decision, action, cost of a wrong call

**Research question:** Given a content item's search and engagement history over the last 90 days,
which pages should a content editor review first this week for a refresh, expansion, or
CTR/engagement fix?

**Decision it improves:** which handful of pages, out of thousands, an editor with limited hours
opens first. Right now that choice is made by whichever rule-of-thumb or spot-check an editor
happens to use — not by a ranked, evidence-backed queue.

**Who acts, and what do they do:** an FlyRank content editor (or account strategist) works down a
ranked review queue. For a page near the top, the action is concrete: rewrite/expand thin content,
fix a title or meta description to lift CTR, or protect a page that is quietly declining before it
falls further. The unit of analysis is one content item (one row = one page, `content_id`),
scored using its own 90-day window of observed signals.

**Cost of a wrong call:**
- *False positive* (page flagged as worth reviewing but isn't): wasted editor hours on a page
  that was fine, or noisy/low-volume, and the editor's time is a fixed, scarce budget.
- *False negative* (a genuinely declining, high-demand page never gets flagged): the page keeps
  losing visibility and traffic uninterrupted until someone happens to notice it by chance -
  a real loss compounding for weeks.
- Because editor time is scarce and both error types are costly in different ways, ranking
  quality (not just "does the model score well") is the thing that matters -
  specifically, how many of the top-K recommendations are actually worth reviewing
  (precision@K), which is exactly what the starter pipeline reports.

**Why data or ML can help at all:** no single column or simple if-statement reliably separates
"worth reviewing" from "fine." Section 3 shows that a naive staleness rule
(`days_since_last_update >= 180` and `impressions_90d >= 500`) catches almost nothing in this
data, while over half the pages are currently trending down and about a third look like clear
CTR underperformers - too many candidates, too little precision, to hand-sort by eye. That is a
pattern that is real but tangled across many weak signals (trend, position, CTR, freshness,
volume) at once, which is exactly the situation where a learned ranking earns its keep over a
fixed rule.

## 3. Quick look at the data (2-3 real numbers)

Loading the starter CSV to back the lane choice above with real counts.

In [1]:
import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

n = len(df)
n_clients = df["client_id"].nunique()

declining_now = (df["trend_direction"] == "down").sum()
declining_with_demand = ((df["trend_direction"] == "down") & (df["impressions_90d"] >= 100)).sum()

low_ctr_visible = (
    (df["impressions_90d"] >= 500)
    & (df["avg_position"] > 0) & (df["avg_position"] <= 20)
    & (df["ctr"] < 0.5)
).sum()

stale_visible = (
    (df["days_since_last_update"] >= 180) & (df["impressions_90d"] >= 500)
).sum()

print(f"rows (one row = one content item): {n:,}")
print(f"unique clients: {n_clients}")
print()
print(f"trend_direction == 'down' (any volume):            {declining_now:>6,}  ({declining_now/n:.1%} of rows)")
print(f"declining AND impressions_90d >= 100 ('real' demand): {declining_with_demand:>6,}  ({declining_with_demand/n:.1%} of rows)")
print(f"low-CTR visible pages (>=500 impr, pos 1-20, ctr<0.5): {low_ctr_visible:>6,}  ({low_ctr_visible/n:.1%} of rows)")
print(f"naive staleness rule (>=180d stale AND >=500 impr):    {stale_visible:>6,}  ({stale_visible/n:.1%} of rows)")

rows (one row = one content item): 30,000
unique clients: 32

trend_direction == 'down' (any volume):            16,262  (54.2% of rows)
declining AND impressions_90d >= 100 ('real' demand): 13,152  (43.8% of rows)
low-CTR visible pages (>=500 impr, pos 1-20, ctr<0.5):  9,759  (32.5% of rows)
naive staleness rule (>=180d stale AND >=500 impr):        17  (0.1% of rows)


## 4. Careful words: what I can and can't claim

**What this work will be able to say:**
- Observed: "in this 90-day window, X% of pages showed pattern Y."
- Associational: "pages with low CTR at a decent position are associated with under-performance
  relative to their impression volume."
- Decision-support: "these pages rank near the top of the review queue, based on evidence in
  their signal history, and are worth a human editor's attention first."
- Validated-ranking: once I have client-holdout validation, "the ranking achieves
  precision@K of ___ on held-out clients" — a claim about the ranking, not about causation.

**What it will never say:**
- Never that a refresh *causes* recovery — this is observational, cross-sectional data with no
  experiment or intervention design behind it. That would need an actual before/after test.
- Never that the model "predicts Google's algorithm" or reverse-engineers ranking factors -
  it only ranks pages within FlyRank's own observed signals.
- Never treat a single low-volume or short-window number as proof - I'll report sample sizes
  and require minimum volume thresholds before flagging anything as a real pattern.
- Never use a product's own rule-based flag (e.g. `health_score`) as a feature or a target for
  "discovery" - the starter data deliberately ships only observable signals, not product
  decisions, to avoid this trap.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.